In [0]:
catalog = "workspace"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.chromatography")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.chromatography.raw_files")
spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA chromatography")

vol_path = f"/Volumes/{catalog}/chromatography/raw_files"

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

random.seed(7)
np.random.seed(7)

instruments = ["HPLC-01", "HPLC-02", "GC-01", "GC-02"]
compounds = ["Caffeine", "Paracetamol", "Ibuprofen", "Aspirin", "Naproxen"]
base_conc = {"Caffeine": 12, "Paracetamol": 8, "Ibuprofen": 5, "Aspirin": 6, "Naproxen": 4}

start = datetime(2026, 5, 1, 8, 0)
rows = []

for _ in range(4000):
    comp = random.choice(compounds)
    unit = random.choices(["mg/L", "ug/L"], weights=[0.7, 0.3])[0]
    conc = abs(np.random.normal(base_conc[comp], base_conc[comp] * 0.2))
    if unit == "ug/L":
        conc = round(conc * 1000, 1)   # same physical amount, just a different unit
    else:
        conc = round(conc, 3)
    ts = start + timedelta(minutes=random.randint(0, 60 * 24 * 30))

    if random.random() < 0.5:
        measured_at = ts.strftime("%Y-%m-%d %H:%M:%S")
    else:
        measured_at = ts.strftime("%m/%d/%Y %H:%M")

    rows.append({
        "instrument_id": random.choice(instruments),
        "sample_id": f"S{random.randint(10000, 99999)}",
        "batch_id": f"B{ts.strftime('%Y%m%d')}-{random.randint(1, 6)}",
        "compound": comp,
        "retention_time_min": round(abs(np.random.normal(3.5, 0.8)), 2),
        "peak_area": int(max(0, base_conc[comp] * 5000 + np.random.normal(0, 8000))),
        "concentration": conc,
        "unit": unit,
        "measured_at": measured_at,
    })

raw = pd.DataFrame(rows)

raw.loc[raw.sample(frac=0.02, random_state=1).index, "concentration"] = np.nan
raw.loc[raw.sample(frac=0.01, random_state=2).index, "peak_area"] = -1
raw = pd.concat([raw, raw.sample(frac=0.03, random_state=3)], ignore_index=True)

raw.to_csv(f"{vol_path}/export_2026_05.csv", index=False)
print(raw.shape)
raw.head()

(4120, 9)


,instrument_id,sample_id,batch_id,compound,retention_time_min,peak_area,concentration,unit,measured_at
0,HPLC-01,S80239,B20260519-1,Ibuprofen,3.13,25262,6690.500,ug/L,05/19/2026 07:15
1,HPLC-01,S66838,B20260524-4,Ibuprofen,2.87,25016,5.408,mg/L,2026-05-24 10:15:00
2,HPLC-01,S39260,B20260526-6,Caffeine,2.10,68141,11.998,mg/L,2026-05-26 09:53:00
3,HPLC-01,S38977,B20260527-1,Naproxen,3.00,18627,4480.400,ug/L,05/27/2026 14:21
4,HPLC-01,S84830,B20260514-3,Naproxen,3.29,18058,4404.200,ug/L,2026-05-14 12:19:00


In [0]:
bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{vol_path}/export_2026_05.csv")
)

bronze.write.mode("overwrite").saveAsTable("bronze_runs")
display(spark.table("bronze_runs").limit(10))

instrument_id,sample_id,batch_id,compound,retention_time_min,peak_area,concentration,unit,measured_at
HPLC-01,S80239,B20260519-1,Ibuprofen,3.13,25262,6690.5,ug/L,05/19/2026 07:15
HPLC-01,S66838,B20260524-4,Ibuprofen,2.87,25016,5.408,mg/L,2026-05-24 10:15:00
HPLC-01,S39260,B20260526-6,Caffeine,2.1,68141,11.998,mg/L,2026-05-26 09:53:00
HPLC-01,S38977,B20260527-1,Naproxen,3.0,18627,4480.4,ug/L,05/27/2026 14:21
HPLC-01,S84830,B20260514-3,Naproxen,3.29,18058,4404.2,ug/L,2026-05-14 12:19:00
HPLC-02,S58810,B20260509-1,Naproxen,3.94,20991,2837.4,ug/L,2026-05-09 13:24:00
HPLC-02,S75066,B20260527-6,Naproxen,2.28,33205,4219.6,ug/L,2026-05-27 00:26:00
GC-02,S57393,B20260515-3,Naproxen,3.19,36232,4.123,mg/L,2026-05-15 15:07:00
GC-01,S78838,B20260512-4,Paracetamol,2.34,36758,7927.4,ug/L,2026-05-12 10:37:00
HPLC-01,S25475,B20260514-5,Ibuprofen,4.34,21668,2711.7,ug/L,05/14/2026 10:30


In [0]:
from pyspark.sql import functions as F

bronze = spark.table("bronze_runs")

measured_at = F.coalesce(
    F.expr("try_to_timestamp(measured_at, 'yyyy-MM-dd HH:mm:ss')"),
    F.expr("try_to_timestamp(measured_at, 'MM/dd/yyyy HH:mm')"),
)

silver = (
    bronze
    .withColumn("measured_at", measured_at)
    .withColumn(
        "concentration_mg_l",
        F.when(F.col("unit") == "ug/L", F.col("concentration") / 1000)
         .otherwise(F.col("concentration"))
    )
    .drop("concentration", "unit")
    .filter(F.col("concentration_mg_l").isNotNull())
    .filter(F.col("peak_area") > 0)
    .dropDuplicates()
)

silver.write.mode("overwrite").saveAsTable("silver_runs")
print(bronze.count(), "->", spark.table("silver_runs").count())

4120 -> 3872


In [0]:
display(spark.table("silver_runs"))

instrument_id,sample_id,batch_id,compound,retention_time_min,peak_area,measured_at,concentration_mg_l
HPLC-01,S38977,B20260527-1,Naproxen,3.0,18627,2026-05-27T14:21:00.000Z,4.4803999999999995
GC-02,S57393,B20260515-3,Naproxen,3.19,36232,2026-05-15T15:07:00.000Z,4.123
GC-01,S54580,B20260527-6,Caffeine,1.85,54702,2026-05-27T09:53:00.000Z,13.285
HPLC-01,S29811,B20260509-5,Caffeine,4.34,64527,2026-05-09T01:41:00.000Z,12.7195
HPLC-02,S74132,B20260529-3,Aspirin,4.09,20616,2026-05-29T21:41:00.000Z,6.023
GC-02,S37503,B20260521-2,Caffeine,4.61,63996,2026-05-21T18:55:00.000Z,11.9688
HPLC-01,S57865,B20260524-2,Naproxen,3.16,1558,2026-05-24T11:41:00.000Z,3.5452
HPLC-01,S35656,B20260516-6,Ibuprofen,3.88,16188,2026-05-16T17:29:00.000Z,4.911
GC-01,S78738,B20260506-5,Aspirin,2.91,51108,2026-05-06T10:28:00.000Z,7.991
GC-01,S10494,B20260515-6,Ibuprofen,4.57,19367,2026-05-15T17:41:00.000Z,3.3817


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_compound_summary AS
SELECT
  instrument_id,
  compound,
  count(*)                              AS run_count,
  round(avg(concentration_mg_l), 3)     AS avg_conc_mg_l,
  round(stddev(concentration_mg_l), 3)  AS stddev_conc_mg_l,
  round(min(retention_time_min), 2)     AS min_rt,
  round(max(retention_time_min), 2)     AS max_rt
FROM silver_runs
GROUP BY instrument_id, compound
""")

display(spark.table("gold_compound_summary"))

instrument_id,compound,run_count,avg_conc_mg_l,stddev_conc_mg_l,min_rt,max_rt
HPLC-01,Caffeine,184,12.0,2.469,1.41,5.64
GC-01,Ibuprofen,176,4.862,0.943,0.67,5.75
GC-01,Aspirin,186,5.843,1.168,1.69,5.65
GC-02,Paracetamol,183,8.044,1.613,1.7,5.69
HPLC-01,Naproxen,201,3.972,0.812,1.03,5.89
GC-01,Paracetamol,207,7.919,1.579,1.78,5.68
HPLC-02,Caffeine,219,12.099,2.268,1.15,5.2
GC-02,Naproxen,186,4.045,0.807,1.04,5.24
GC-01,Naproxen,191,4.056,0.857,1.04,5.42
HPLC-02,Ibuprofen,177,4.988,1.002,1.06,5.38


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_qc AS
WITH stats AS (
  SELECT compound,
         avg(concentration_mg_l)    AS mu,
         stddev(concentration_mg_l) AS sd
  FROM silver_runs
  GROUP BY compound
)
SELECT
  s.instrument_id,
  s.compound,
  s.sample_id,
  s.measured_at,
  s.concentration_mg_l,
  CASE WHEN abs(s.concentration_mg_l - st.mu) > 3 * st.sd
       THEN 'review' ELSE 'pass' END AS qc_status
FROM silver_runs s
JOIN stats st USING (compound)
""")

display(spark.sql("SELECT qc_status, count(*) FROM gold_qc GROUP BY qc_status"))

qc_status,count(*)
pass,3861
review,11
